# XGBoost — SoH Estimation

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import json

from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

work_dir = os.getcwd()
print("work_dir:", work_dir)
MODEL_NAME = "xgboost"


## Data Loader

In [ ]:
input_folder = os.path.join(work_dir, "data_normalized")

X_train = np.load(os.path.join(input_folder, "X_train.npy"))
y_train = np.load(os.path.join(input_folder, "y_train.npy"))
X_val = np.load(os.path.join(input_folder, "X_val.npy"))
y_val = np.load(os.path.join(input_folder, "y_val.npy"))
X_test = np.load(os.path.join(input_folder, "X_test.npy"))
y_test = np.load(os.path.join(input_folder, "y_test.npy"))


def flatten(X):
    return X.reshape(X.shape[0], -1)


X_train_f = flatten(X_train)
X_val_f = flatten(X_val)
X_test_f = flatten(X_test)
print(f"Train: {X_train_f.shape} | Val: {X_val_f.shape} | Test: {X_test_f.shape}")


## Config (best Optuna params)

In [ ]:
# Best params from Optuna
CONFIG = {
    "n_estimators": 1544,
    "max_depth": 3,
    "learning_rate": 0.01322,
    "subsample": 0.9338,
    "colsample_bytree": 0.7776,
    "min_child_weight": 4,
    "gamma": 1.3329,
    "reg_alpha": 0.9945,
    "reg_lambda": 0.9342,
    "tree_method": "hist",
    "random_state": 42,
    "eval_metric": "rmse",
    "early_stopping_rounds": 50,
}


## Train

In [ ]:
model = XGBRegressor(**CONFIG)
model.fit(
    X_train_f, y_train, eval_set=[(X_train_f, y_train), (X_val_f, y_val)], verbose=50
)


## Predict

In [ ]:
y_pred = model.predict(X_test_f)

## Metrics & Figures

In [ ]:
output_figure = os.path.join(work_dir, "figures", MODEL_NAME)
os.makedirs(output_figure, exist_ok=True)
residuals = y_test - y_pred

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"RMSE : {rmse:.4f} %")
print(f"MAE  : {mae:.4f} %")
print(f"R²   : {r2:.6f}")

# 1 – True vs Pred
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("True SoH")
plt.ylabel("Predicted SoH")
plt.title("Prediction vs Ground Truth")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig1_true_vs_pred.png"))
plt.show()

# 2 – Residual
plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, alpha=0.3)
plt.axhline(0, color="r")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig2_residuals.png"))
plt.show()

# 3 – Residual distribution
plt.figure(figsize=(6, 4))
sns.histplot(residuals, bins=50, kde=True)
plt.title("Residual Distribution")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig3_residual_dist.png"))
plt.show()

# 4 – Error vs SoH
plt.figure(figsize=(6, 4))
plt.scatter(y_test, np.abs(residuals), alpha=0.3)
plt.xlabel("True SoH")
plt.ylabel("Absolute Error")
plt.title("Error vs SoH")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig4_error_vs_soh.png"))
plt.show()

# 5 – Sorted
idx = np.argsort(y_test)
plt.figure(figsize=(8, 4))
plt.plot(y_test[idx], label="True")
plt.plot(y_pred[idx], label="Predicted", alpha=0.6)
plt.legend()
plt.title("Sorted SoH Prediction")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig5_sorted_pred.png"))
plt.show()

# 6 – Training curve (XGBoost eval result)
evals = model.evals_result()
plt.figure(figsize=(6, 4))
plt.plot(evals["validation_0"]["rmse"], label="train")
plt.plot(evals["validation_1"]["rmse"], label="val")
plt.legend()
plt.title("Training Curve (RMSE)")
plt.xlabel("Boosting round")
plt.ylabel("RMSE")
plt.grid()
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig6_train_curve.png"))
plt.show()

# 7 – Violin + box
plt.figure(figsize=(8, 5))
sns.violinplot(x=np.abs(residuals), inner="quartile")
sns.boxplot(x=np.abs(residuals), width=0.2, color="white")
plt.title("Error Distribution (Violin + Box)")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig7_violin_box.png"))
plt.show()

# 8 – CDF
errors = np.abs(residuals)
sorted_err = np.sort(errors)
cdf = np.arange(len(errors)) / len(errors)
plt.figure()
plt.plot(sorted_err, cdf)
plt.xlabel("Absolute Error")
plt.ylabel("CDF")
plt.title("Error CDF")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig8_cdf.png"))
plt.show()

# 9 – Histogram KDE
plt.figure(figsize=(8, 5))
sns.histplot(np.abs(residuals), bins=50, kde=True)
plt.title("Error Distribution (Histogram + KDE)")
plt.xlabel("Absolute Error")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig9_histogram.png"))
plt.show()

# 10 – Log scale
plt.figure(figsize=(8, 5))
sns.histplot(np.abs(residuals), bins=50, log_scale=(False, True))
plt.title("Error Distribution (Log Scale)")
plt.xlabel("Absolute Error")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig10_logscale.png"))
plt.show()

print("MAE    :", np.mean(np.abs(residuals)))
print("RMSE   :", np.sqrt(np.mean(residuals**2)))
print("Max Err:", np.max(np.abs(residuals)))
print("Median :", np.median(np.abs(residuals)))


## Save Model

In [ ]:
model_dir = os.path.join(work_dir, "model")
os.makedirs(model_dir, exist_ok=True)
joblib.dump(model, os.path.join(model_dir, "xgboost.pkl"))
print("Saved")